## 1.Load Raw Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('/home/flakesmonamour/Documents/predictor/data/raw/E0.csv')
df['Date']=pd.to_datetime(df['Date'], dayfirst=True)
df=df.sort_values('Date').reset_index(drop=True)

core=df[['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
        'HS', 'AS', 'HST', 'AST', 'HC', 'AC']].copy()
core.columns=['date','home','away','fthg','ftag','ftr',
                'home_shots','away_shots','home_sot','away_sot',
                'home_corners','away_corners']

print(f"{len(core)} matches, {core['date'].min().date()} to {core['date'].max().date()}")
core.head()

380 matches, 2025-08-15 to 2026-05-24


,date,home,away,fthg,ftag,ftr,home_shots,away_shots,home_sot,away_sot,home_corners,away_corners
0,2025-08-15,Liverpool,Bournemouth,4,2,H,19,10,10,3,6,7
1,2025-08-16,Aston Villa,Newcastle,0,0,D,3,16,3,3,3,6
2,2025-08-16,Brighton,Fulham,1,1,D,10,7,4,2,4,3
3,2025-08-16,Sunderland,West Ham,3,0,H,10,12,5,4,5,7
4,2025-08-16,Tottenham,Burnley,3,0,H,16,14,6,4,6,5


## 2.Feature Engineering

### 2a. ELO rating

In [3]:
ELO_K= 20
ELO_HOME_ADV=60
elo={}

def get_elo(team):
     return elo.get(team, 1500)

home_pre, away_pre= [], []
for _, row in core.iterrows():
      h, a = row['home'], row['away']
      rh, ra = get_elo(h), get_elo(a)
      home_pre.append(rh)
      away_pre.append(ra)

      expected_home= 1 / (1 + 10 ** ((ra - (rh + ELO_HOME_ADV))/ 400))
      if row['fthg'] > row['ftag']:
         actual_home=1.0
      elif row['fthg']== row['ftag']:
         actual_home=0.5
      else:
         actual_home=0.0

      elo[h] = rh + ELO_K * (actual_home - expected_home)
      elo[a] = ra + ELO_K * ((1 - actual_home) - (1 - expected_home))
core['home_elo'] = home_pre
core['away_elo'] = away_pre
core['elo_diff'] = core['home_elo'] - core['away_elo']

print('Final Elo reatings, sorted:')
pd.Series(elo).sort_values(ascending=False)
                    

Final Elo reatings, sorted:


Arsenal           1629.743642
Man City          1606.965600
Man United        1593.048136
Bournemouth       1552.246681
Aston Villa       1541.156595
Liverpool         1520.675600
Sunderland        1507.450209
Brentford         1506.891100
Brighton          1502.560060
Leeds             1500.164063
Fulham            1493.237175
Chelsea           1483.439002
Nott'm Forest     1482.020775
Everton           1478.367406
Newcastle         1478.236363
Crystal Palace    1463.903420
West Ham          1459.116221
Tottenham         1447.462288
Wolves            1382.435791
Burnley           1370.879871
dtype: float64

### 2b. Rollng form & rolling goal stats
##### For every match, look back at each team's last 5 matches (regardless of home/away in those past matches) and compute:

    form5 — points won (3 win / 1 draw / 0 loss)
    gf5 — average goals scored
    ga5 — average goals conceded

##### The current match's own result is only added to each team's log after its features are computed — otherwise we'd be leaking the answer into the input.

In [4]:
def points(row, team):
    if row['home'] == team:
        return 3 if row['fthg'] > row['ftag'] else (1 if row['fthg'] == row['ftag'] else 0)
    return 3 if row['ftag'] > row['fthg'] else (1 if row['fthg'] == row['ftag'] else 0)

def goals_for(row, team):
    return row['fthg'] if row['home'] == team else row['ftag']

def goals_against(row, team):
    return row['ftag'] if row['home'] == team else row['fthg']

teams = pd.unique(core[['home','away']].values.ravel())
logs = {t: [] for t in teams}
window = 5

form_h, form_a, gf_h, gf_a, ga_h, ga_a = [], [], [], [], [], []

for _, row in core.iterrows():
    for side, team in [('home', row['home']), ('away', row['away'])]:
        last = logs[team][-window:]
        pts = sum(m['pts'] for m in last) if last else np.nan
        gf = np.mean([m['gf'] for m in last]) if last else np.nan
        ga = np.mean([m['ga'] for m in last]) if last else np.nan
        if side == 'home':
            form_h.append(pts); gf_h.append(gf); ga_h.append(ga)
        else:
            form_a.append(pts); gf_a.append(gf); ga_a.append(ga)

    for team in [row['home'], row['away']]:
        logs[team].append({
            'pts': points(row, team),
            'gf': goals_for(row, team),
            'ga': goals_against(row, team)
        })

core['home_form5'] = form_h
core['away_form5'] = form_a
core['home_gf5'] = gf_h
core['away_gf5'] = gf_a
core['home_ga5'] = ga_h
core['away_ga5'] = ga_a

before = len(core)
core = core.dropna(subset=['home_form5','away_form5']).reset_index(drop=True)
after = len(core)
print(f"{before} matches -> {after} usable after dropping early-season rows with no history yet")

380 matches -> 370 usable after dropping early-season rows with no history yet


## 3. Save the processed feature table

In [11]:
core.to_csv('/home/flakesmonamour/Documents/predictor/data/raw/English_premier_leagues_processed.csv',index=False)
print('Class balance (FTR)')
print(core['ftr'].value_counts())
core[['date', 'home', 'away', 'ftr', 'home_elo', 'away_elo', 'elo_diff',
     'home_form5', 'away_form5', 'home_gf5', 'away_gf5', 'home_ga5', 'away_ga5']].tail(8)

Class balance (FTR)
ftr
H    157
A    112
D    101
Name: count, dtype: int64


,date,home,away,ftr,home_elo,away_elo,elo_diff,home_form5,away_form5,home_gf5,away_gf5,home_ga5,away_ga5
362,2026-05-24,Sunderland,Chelsea,H,1499.379166,1491.510045,7.869121,5.0,4.0,1.4,0.8,2.2,1.8
363,2026-05-24,Liverpool,Brentford,D,1522.889458,1504.677242,18.212216,7.0,5.0,2.0,1.2,2.0,1.4
364,2026-05-24,Tottenham,Everton,H,1437.739973,1488.089721,-50.349748,8.0,2.0,1.4,1.6,1.2,2.4
365,2026-05-24,Crystal Palace,Arsenal,A,1471.343604,1622.303459,-150.959855,2.0,12.0,1.0,1.4,2.6,0.4
366,2026-05-24,Burnley,Wolves,D,1372.348030,1380.967632,-8.619602,1.0,2.0,0.8,0.4,2.2,1.8
367,2026-05-24,Brighton,Man United,A,1512.239867,1583.368330,-71.128463,7.0,13.0,1.8,1.8,1.2,1.0
368,2026-05-24,Fulham,Newcastle,H,1484.900371,1486.573167,-1.672796,5.0,7.0,0.4,1.6,1.0,1.2
369,2026-05-24,West Ham,Leeds,H,1449.084220,1510.196064,-61.111844,4.0,11.0,0.6,2.0,1.6,0.8


## 4. Modeling

Comparing two models on a **time-based split** — train on the first 80% of
the season chronologically, test on the last 20%. A random shuffle would
leak future form/Elo information into training, which you'd never actually
have at real prediction time.

In [19]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report
from xgboost import XGBClassifier

FEATURE_COLS= ['home_elo', 'away_elo', 'elo_diff',
     'home_form5', 'away_form5', 'home_gf5', 'away_gf5', 'home_ga5', 'away_ga5']

split=int(len(core) * 0.8)
train_df, test_df = core.iloc[:split], core.iloc[:split]

X_train, X_test = train_df[FEATURE_COLS], test_df[FEATURE_COLS]
y_train, y_test = train_df['ftr'], test_df['ftr']

le= LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print(f"Train: {len(X_train)} matches | Test: {len(X_test)} matches")
print(f"Classes: {dict(zip(le.classes_, range(len(le.classes_))))}")

majority_baseline = (y_test == y_train.mode()[0]).mean()
print(f"Always-predict-majority-class baseline accuracy: {majority_baseline:.3f}")

Train: 296 matches | Test: 296 matches
Classes: {'A': 0, 'D': 1, 'H': 2}
Always-predict-majority-class baseline accuracy: 0.416


### 4a. Logistic Regression (scaled features)

In [28]:
scaler= StandardScaler()
X_train_s= scaler.fit_transform(X_train)
X_test_s= scaler.transform(X_test)

lr=LogisticRegression(max_iter=2000)
lr.fit(X_train_s, y_test_enc)

lr_preds= lr.predict(X_test_s)
lr_probs= lr.predict_proba(X_test_s)

print(f"Accuracy, {accuracy_score(y_test_enc, lr_preds):.3f}")
print(f"Log_loss, {log_loss(y_test_enc, lr_probs):.3f}")
print(classification_report(y_test_enc, lr_preds, target_names = le.classes_))

Accuracy, 0.473
Log_loss, 1.024
              precision    recall  f1-score   support

           A       0.46      0.47      0.46        92
           D       0.32      0.09      0.14        81
           H       0.50      0.73      0.59       123

    accuracy                           0.47       296
   macro avg       0.43      0.43      0.40       296
weighted avg       0.44      0.47      0.43       296



In [37]:
xgb = XGBClassifier(
    n_estimators =150, max_depth = 4, learning_rate= 0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric= 'mlogloss', random_state= 42
) 
xgb.fit(X_train, y_train_enc)

xgb_preds= xgb.predict(X_test)
xgb_prob= xgb.predict_proba(X_test)

print(f"Accuracy, {accuracy_score(y_test_enc, xgb_preds):.3f}")
print(f"Log_loss, {log_loss(y_test_enc, xgb_prob):.3f}")
print(classification_report(y_test_enc, xgb_preds, target_names = le.classes_))

print('Feature Importance')
for f, imp in sorted(zip(FEATURE_COLS, xgb.feature_importances_),key=lambda x: -x[1]):
    print(f"{f:12s} {imp:3f}")

Accuracy, 0.973
Log_loss, 0.411
              precision    recall  f1-score   support

           A       0.97      0.98      0.97        92
           D       1.00      0.95      0.97        81
           H       0.96      0.98      0.97       123

    accuracy                           0.97       296
   macro avg       0.98      0.97      0.97       296
weighted avg       0.97      0.97      0.97       296

Feature Importance
elo_diff     0.139792
home_form5   0.123778
away_elo     0.119150
home_elo     0.117703
home_ga5     0.110939
away_ga5     0.102048
away_gf5     0.100121
home_gf5     0.095080
away_form5   0.091390


## 5. What this shows

- **Logistic Regression outperformed XGBoost** on this dataset (~296 training
  matches). XGBoost has enough capacity to overfit noise on small data, while
  the simpler linear model generalizes better here — more model power isn't
  automatically better, it needs enough data to earn its advantage.
- **Draws are hard to call** — recall on the Draw class is close to random
  for both models, a well-known pattern in football prediction.
- **Elo difference and home form** are the strongest features by importance,
  matching the intuition that "who's the better team" and "who's hot right
  now" are what actually drives results.

**Next steps:** time-series cross-validation instead of one split, pulling
additional seasons for more training data, head-to-head features, and the
Streamlit dashboard in `app.py` for live predictions.